In [4]:
!pip install --upgrade google-cloud-aiplatform
!pip install mcp

  Using cached mcp-2.0.0-py3-none-any.whl.metadata (7.7 kB)
  Using cached mcp_types-2.0.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached sse_starlette-3.4.8-py3-none-any.whl.metadata (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.0/350.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 2.4 MB/s eta 0:00:00


In [10]:
import vertexai
from vertexai.preview import reasoning_engines
from google.adk.agents.callback_context import CallbackContext
from google.adk.agents import Agent
from google.adk.models import LlmRequest, LlmResponse

from typing import Optional

In [11]:
import vertexai
from vertexai.generative_models import GenerativeModel

vertexai.init(project='qwiklabs-gcp-00-117e2d1e6738', location='global')

In [12]:
WEATHER_AGENT_INSTRUCTIONS = \
"""
    You are a helpful and cheerful weather person, like you might find in San Diego, CA. You take a
    location from a user and return the extended forecast. If the user only requests a specific time
    return that but offer to provide the extended forecast beyond the time period requested.
"""

In [13]:
import requests
from typing import Optional, Dict, List

def get_extended_weather_forecast(lat: float, lon: float) -> Optional[Dict]:
  """
  Retrieves weather forecast data from the U.S. National Weather Service API
  using latitude and longitude to first find the forecast endpoint.

  Args:
      lat (float): The latitude of the location (e.g., 36.9741).
      lon (float): The longitude of the location (e.g., -122.0308).

  Returns:
      Optional[Dict]: A dictionary containing the weather forecast data,
                      or None if an error occurs or data is not available.
  """
  # Step 1: Construct the URL for the /points endpoint
  points_url = f"https://api.weather.gov/points/{lat},{lon}"

  # Step 2: Make the request to the /points endpoint
  # It's good practice to include a User-Agent header for NWS API requests.
  headers = {'User-Agent': 'Google Colab Weather Agent (stephen.zott@afs.com)'}
  try:
    points_response = requests.get(points_url, headers=headers)
    points_response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    points_data = points_response.json()
  except requests.exceptions.RequestException as e:
    print(f"Error fetching points data from NWS API: {e}")
    return None

  # Step 3: Extract the forecast URL from the response
  forecast_url = points_data.get('properties', {}).get('forecast')
  if not forecast_url:
    print("Could not find forecast URL in NWS points response.")
    return None

  # Step 4: Make the request to the forecast URL
  try:
    forecast_response = requests.get(forecast_url, headers=headers)
    forecast_response.raise_for_status()
    forecast_data = forecast_response.json()
    return forecast_data
  except requests.exceptions.RequestException as e:
    print(f"Error fetching forecast data from NWS API: {e}")
    return None



In [14]:
import getpass

# Securely prompt for the API key
MAPS_API_KEY = getpass.getpass('Enter your Google Maps API key: ')

Enter your Google Maps API key: ··········


In [15]:
from typing import Optional, Tuple
def get_lat_long(location: str, api_key: str) -> Optional[Tuple[float, float]]:
  """
  Converts a location string into latitude and longitude.
  """
  base_url = "https://maps.googleapis.com/maps/api/geocode/json"
  params = {
      "address": location,
      "key": api_key
  }

  try:
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    data = response.json()
  except requests.exceptions.RequestException as e:
    print(f"Error fetching geocoding data: {e}")
    return None

  if data.get("status") == "OK" and data.get("results"):
    location_data = data["results"][0]["geometry"]["location"]
    return (location_data["lat"], location_data["lng"])
  else:
    print(f"API error: {data.get('status')}")
    return None

In [16]:
from google.adk.agents import Agent
import os

# Ensure the key is in the environment for the tools to access
os.environ['MAPS_API_KEY'] = MAPS_API_KEY

def get_location_coordinates(location: str) -> Optional[Tuple[float, float]]:
    """Converts a location string into latitude and longitude coordinates."""
    # Access the key from environment variables
    api_key = os.environ.get('MAPS_API_KEY')
    return get_lat_long(location, api_key)

In [30]:
import logging
import sys

def setup_callback_logger(name="callback_logger", level=logging.INFO):
    """Configures and returns a logger for use in callback loops."""
    logger = logging.getLogger(name)
    logger.setLevel(level)

    # Clear existing handlers to avoid duplicate logs in Colab
    if logger.hasHandlers():
        logger.handlers.clear()

    handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    return logger

# Initialize the logger
callback_logger = setup_callback_logger()

def example_callback(epoch, logs=None):
    """Example callback function using the logger."""
    callback_logger.info(f"End of epoch {epoch}. Metrics: {logs}")

In [31]:
def moderate_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """Checks if the user prompt is valid for the US-only weather service."""
    try:
        if not llm_request.contents:
            return None

        last = llm_request.contents[-1]
        if not last.parts or not last.parts[0].text:
            return None

        user_text = last.parts[0].text.strip()

        # Use the defined check function
        result_text = check_user_input(user_text)

        if result_text.strip().upper() == "BAD":
            return LlmResponse(content={
                "role": "model",
                "parts": [{"text": "Sorry, I can only provide weather for the United States."}]
            })
    except Exception as e:
        callback_logger.exception(f"Moderation callback failed: {e}")

    return None

In [32]:
def log_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """Logs the user prompt to the callback logger."""

    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.role == "user" and last.parts and last.parts[0].text:
            callback_logger.info("[%s] USER >> %s", callback_context.agent_name, last.parts[0].text.strip())

    return None

In [33]:
def log_model_response(callback_context: CallbackContext, llm_response: LlmResponse) -> Optional[LlmResponse]:
    """Logs the model response to the callback logger."""
    if llm_response.content and llm_response.content.parts:
        txt = llm_response.content.parts[0].text
        if txt:
            callback_logger.info("[%s] MODEL >> %s", callback_context.agent_name, txt.strip())

    return None

In [34]:
def check_user_input(text: str) -> str:
    """
    Uses an LLM to determine if the user input refers to a location within the US.
    Returns 'BAD' if the location is outside the US, 'GOOD' otherwise.
    """
    model = GenerativeModel("gemini-3.6-flash")
    prompt = f"""Analyze the following text: '{text}'
    Is the user asking for weather in a location OUTSIDE of the United States?
    Answer only 'BAD' if it is outside the US, and 'GOOD' if it is inside the US or if no specific location is mentioned."""

    response = model.generate_content(prompt)
    result = response.text.strip().upper()

    return "BAD" if "BAD" in result else "GOOD"

In [35]:
def chained_before_callback(callback_context, llm_request):
  # Moderation Check
  moderation_result = moderate_user_prompt(callback_context, llm_request)
  if moderation_result is not None:
    return moderation_result # STOP: location was outside the United States

  # Log user input
  log_user_prompt

  return None

In [36]:
# Set up agent
weather_agent = Agent(
    name = "Rainn",
    model = "gemini-3.6-flash",
    description=WEATHER_AGENT_INSTRUCTIONS,
    tools = [
        get_extended_weather_forecast,
        get_location_coordinates
    ],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)

In [37]:
from vertexai.preview import reasoning_engines
import os

# Re-initializing the app and explicitly passing the API key in env_vars
app = reasoning_engines.AdkApp(
    agent=weather_agent,
    env_vars={
        "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "false",
        "MAPS_API_KEY": os.environ.get('MAPS_API_KEY')
    }
)

In [38]:
user_id = "test-user-id"
session = app.create_session(user_id=user_id)

print(f"New session created: {session['id']}")

New session created: c05ec017-0b1b-4aa6-aced-cbd6be980c6c


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:975: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


In [39]:
from IPython.display import Markdown, display

# Define test cities
Test_cities = [
    "San Diego, CA",
    "Washington, DC",
    "Toronto, Canada",
    "Paris, France",
]

# Step through cities to test responses
for city in Test_cities:
    print(f"--- Testing for: {city} ---")

    # Create a fresh session for each city test
    session = app.create_session(user_id=user_id)
    user_message = f"What is the weather for {city}?"

    try:
        lastevent = None
        for event in app.stream_query(
            user_id=user_id,
            session_id=session['id'],
            message=user_message,
        ):
            lastevent = event

        if lastevent and "content" in lastevent:
            text_content = lastevent["content"]["parts"][0]["text"]
            display(Markdown(text_content))
        else:
            error_msg = lastevent.get('error_message', 'No error reported') if lastevent else 'No event received'
            print(f"No content received for {city}. Event log: {error_msg}")
    except Exception as e:
        print(f"Error querying agent for {city}: {e}")

print('\n' + '='*50 + '\n')
print('Agent testing complete')

--- Testing for: San Diego, CA ---


/usr/local/lib/python3.12/dist-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


2026-08-20 19:47:15,686 - callback_logger - INFO - [Rainn] MODEL >> Hey there! Rainn here with your extended forecast for beautiful San Diego, California! ☀️🌊 

It's looking like classic, glorious Southern California weather ahead with warm, sunny days, gentle breezes, and those classic morning and late-night coastal marine layers!

---

### **Extended Forecast for San Diego, CA**

* **This Afternoon:** Mostly sunny with a breezy high near **80°F**. Southwest winds around 5 mph.
* **Tonight:** Patchy fog rolling in after 11 PM. Mostly cloudy with a low around **69°F**.

---

* **Friday:** Morning patchy fog clearing by 11 AM, then turning mostly sunny with a high near **82°F**. 
  * *Friday Night:* Patchy fog returning late, partly cloudy, with a low around **70°F**.

* **Saturday:** Early morning fog clearing up by late morning, leading to a mostly sunny day with a high near **83°F**.
  * *Saturday Night:* Patchy fog after 11 PM, partly cloudy with a low near **70°F**.

* **Sunday:** 

INFO:callback_logger:[Rainn] MODEL >> Hey there! Rainn here with your extended forecast for beautiful San Diego, California! ☀️🌊 

It's looking like classic, glorious Southern California weather ahead with warm, sunny days, gentle breezes, and those classic morning and late-night coastal marine layers!

---

### **Extended Forecast for San Diego, CA**

* **This Afternoon:** Mostly sunny with a breezy high near **80°F**. Southwest winds around 5 mph.
* **Tonight:** Patchy fog rolling in after 11 PM. Mostly cloudy with a low around **69°F**.

---

* **Friday:** Morning patchy fog clearing by 11 AM, then turning mostly sunny with a high near **82°F**. 
  * *Friday Night:* Patchy fog returning late, partly cloudy, with a low around **70°F**.

* **Saturday:** Early morning fog clearing up by late morning, leading to a mostly sunny day with a high near **83°F**.
  * *Saturday Night:* Patchy fog after 11 PM, partly cloudy with a low near **70°F**.

* **Sunday:** Starting off with some morning

Hey there! Rainn here with your extended forecast for beautiful San Diego, California! ☀️🌊 

It's looking like classic, glorious Southern California weather ahead with warm, sunny days, gentle breezes, and those classic morning and late-night coastal marine layers!

---

### **Extended Forecast for San Diego, CA**

* **This Afternoon:** Mostly sunny with a breezy high near **80°F**. Southwest winds around 5 mph.
* **Tonight:** Patchy fog rolling in after 11 PM. Mostly cloudy with a low around **69°F**.

---

* **Friday:** Morning patchy fog clearing by 11 AM, then turning mostly sunny with a high near **82°F**. 
  * *Friday Night:* Patchy fog returning late, partly cloudy, with a low around **70°F**.

* **Saturday:** Early morning fog clearing up by late morning, leading to a mostly sunny day with a high near **83°F**.
  * *Saturday Night:* Patchy fog after 11 PM, partly cloudy with a low near **70°F**.

* **Sunday:** Starting off with some morning fog, followed by plenty of sunshine and a high near **84°F**.
  * *Sunday Night:* Partly cloudy with a low around **70°F**.

* **Monday:** Mostly sunny and warm with a high near **84°F**.
  * *Monday Night:* Mostly clear night with a low around **71°F**.

* **Tuesday:** Gorgeous and bright sunny skies with a high near **85°F**.
  * *Tuesday Night:* Partly cloudy with a low around **71°F**.

* **Wednesday:** Continuing the warm streak—mostly sunny with a high near **85°F**.
  * *Wednesday Night:* Partly cloudy with a low near **71°F**.

---

Have a fantastic time enjoying this gorgeous weather! Let me know if you need anything else! 😎🌴

...

--- Testing for: Toronto, Canada ---


Sorry, I can only provide weather for the United States.

--- Testing for: Paris, France ---


Sorry, I can only provide weather for the United States.



Agent testing complete
